# Sionna PHY

> 5G NR link level with Sionna PHY: every bit LDPC-coded with a CRC, QAM, over the channel at each subcarrier.

In [ ]:
#| default_exp sionna_phy

In [ ]:
#| hide
from nbdev.showdoc import *

The link level of 5G NR with [Sionna PHY](https://nvlabs.github.io/sionna/phy/index.html): every bit of every
transmission, coded and decoded the way an NR receiver does it.

1. The payload becomes bits (a `Codec`, e.g. `IndexCodec` for codebook indices).
2. **Coding:** the bits become an NR transport block, with a CRC, LDPC coding and rate matching (`TBEncoder`), at
   the MCS of the transmission.
3. **Modulation:** QAM (`Mapper`).
4. **The channel:** the symbols go out on resource elements, filling the subcarriers in turn. Each sees the
   channel of its subcarrier: `y = h_k √P x + n`, where the noise includes any interference.
5. **The receiver:** it equalizes each resource element (one tap, knowing `h_k`), computes soft bits (`Demapper`,
   LLRs), and decodes the LDPC code (`TBDecoder`).

The CRC decides delivery, as in a real receiver: a transport block that fails its CRC is lost.

`SionnaLinkChannel` is a `RadioChannel`, like `SystemLevelChannel` and `LinkLevelChannel`. It takes the same
fadings (e.g. `RayTracedPathsFading` with subcarriers), power control, `InterferenceBackend` and protocols. It needs
Sionna 2 (PyTorch): `pip install 'comm-core[sionna]'`.

**The MCS:** a fixed `mcs_index`, or link adaptation (the default). Link adaptation picks the highest MCS that
Sionna SYS (`SionnaErrorModel`) predicts to meet `bler_target` at the transmission's SINR. The transmitter is assumed
to know the channel.

**The results** add `mcs`, `resource_elements`, `tb_size` (the NR transport block, padded to an allowed size) and
`bit_errors` to their metadata. `error_rate` is the bit error rate of the decoded payload.

**MCSs:** those with code rates below 1/5 (MCS 0–2 of table 1) are not available. Sionna's LDPC encoder
doesn't support them (NR reaches them by repetition), so link adaptation chooses from the others.

**Cost:** LDPC decoding is heavy, around 25 ms per 1000-bit message on a CPU, much less on a GPU. Use it to check the
system level (`SionnaErrorModel`), or where the bits themselves matter. For training, `SystemLevelChannel` with
`SionnaErrorModel` is the fast equivalent.

In [ ]:
#| export
from __future__ import annotations

import math
from collections import defaultdict
from typing import List, Sequence, Tuple

import numpy as np

from comm_core.core import Message, Transmission, TransmissionResult
from comm_core.link import Codec, IndexCodec
from comm_core.propagation import Fading
from comm_core.sionna_sys import SionnaErrorModel
from comm_core.system import LinkBudget, PowerControl, RadioChannel
from comm_core.topology import Link

In [ ]:
#| export
MIN_LDPC_CODERATE = 0.2                                  # Sionna's 5G LDPC encoder: rates below 1/5 are not supported


class SionnaLinkChannel(RadioChannel):
    "5G NR link level with Sionna PHY: every bit LDPC-coded with a CRC, QAM, over the channel at each subcarrier."

    def __init__(
        self,
        codec: Codec | None = None, # Payload <-> bits; None: `IndexCodec(256)`
        fading: Fading | None = None, # The channel of a transmission (with subcarriers, e.g. `RayTracedPathsFading`)
        power_control: PowerControl | None = None, # Transmit power and outages; None: the link's tx_power_w
        noise_power: float = 1.0, # Noise power at the receiver, per resource element
        mcs_index: int | None = None, # A fixed MCS; None: link adaptation (see `SionnaErrorModel`)
        bler_target: float = 0.1, # Link adaptation's target error rate
        mcs_table_index: int = 1, # 5G NR MCS table: 1 (up to 64-QAM) or 2 (up to 256-QAM)
        mcs_category: int = 0, # 0: PUSCH (uplink), 1: PDSCH (downlink)
        num_bp_iter: int = 20, # LDPC decoder iterations
        seed: int | None = None, # Seed of the noise and the fading
        device: str | None = None, # Torch device; None: Sionna's default (a GPU if there is one)
        precision: str = 'single', # 'single' or 'double'
    ):
        import torch
        from sionna.phy.nr.utils import MCSDecoderNR
        self._torch = torch
        self.codec = codec if codec is not None else IndexCodec()
        decoder, supported = MCSDecoderNR(device=device), []
        for m in range(32):                                # the MCSs Sionna PHY can encode (code rate >= 1/5)
            try:
                _, rate = decoder(torch.tensor([m], dtype=torch.int32), mcs_table_index=mcs_table_index,
                                  mcs_category=mcs_category, check_validity=True)
            except ValueError:
                continue
            if float(rate) >= MIN_LDPC_CODERATE:
                supported.append(m)
        if mcs_index is not None and mcs_index not in supported:
            raise ValueError(f"MCS {mcs_index}: not one Sionna PHY can encode (code rates below 1/5 are not supported); "
                             f"one of {supported}")
        self.adaptation = SionnaErrorModel(mcs_index=mcs_index, bler_target=bler_target, mcs_table_index=mcs_table_index,
                                           mcs_category=mcs_category, device=device, precision=precision,
                                           mcs_indices=supported)
        self.device, self.precision = self.adaptation.device, precision
        self.real = torch.float64 if precision == 'double' else torch.float32
        self.complex = torch.complex128 if precision == 'double' else torch.complex64
        self.mcs_table_index, self.mcs_category, self.num_bp_iter = mcs_table_index, mcs_category, num_bp_iter
        self._links = {}                                  # (mcs, bits) -> the coding chain, built once
        super().__init__(fading, power_control, noise_power, seed)

    def _link(self, mcs: int, num_bits: int) -> tuple:
        "(bits per symbol, resource elements, encoder, decoder, mapper, demapper) of an MCS and a payload size."
        if (mcs, num_bits) not in self._links:
            import contextlib, io
            torch = self._torch
            from sionna.phy.mapping import Demapper, Mapper
            from sionna.phy.nr import TBDecoder, TBEncoder
            i = self.adaptation.mcs.cpu().tolist().index(mcs)
            q, r = self.adaptation.modulation_order[i], self.adaptation.coderate[i]
            num_re, tb_size = self.adaptation.allocation(num_bits, mcs)     # the same allocation as the system level
            with contextlib.redirect_stdout(io.StringIO()):
                encoder = TBEncoder(target_tb_size=tb_size, num_coded_bits=num_re * q, target_coderate=r,
                                    num_bits_per_symbol=q, channel_type='PUSCH' if self.mcs_category == 0 else 'PDSCH',
                                    precision=self.precision, device=self.device)
            self._links[mcs, num_bits] = (q, num_re, encoder,
                                         TBDecoder(encoder, num_bp_iter=self.num_bp_iter, precision=self.precision, device=self.device),
                                         Mapper('qam', q, precision=self.precision, device=self.device),
                                         Demapper('app', 'qam', q, precision=self.precision, device=self.device))
        return self._links[mcs, num_bits]

    def _num_bits(self, message: Message) -> int:
        return int(np.size(self.codec.encode(message.payload)))

    def transmit(
        self,
        transmission: Transmission,
        link: Link,
        interference: float = 0.0, # Power of the other transmissions at the receiver (see `InterferenceBackend`)
        budget: LinkBudget | None = None, # The transmission's link budget, if already computed
    ) -> TransmissionResult:
        return self.transmit_batch([(transmission, link)], [interference], [budget])[0]

    def transmit_batch(
        self,
        transmissions: List[Tuple[Transmission, Link]], # Transmissions, with their links
        interference: Sequence[float] | None = None, # Power of the other transmissions at each receiver
        budgets: Sequence[LinkBudget | None] | None = None, # Their link budgets, if already computed
    ) -> List[TransmissionResult]:
        "Several transmissions: the ones with the same MCS and size are coded and decoded together."
        torch = self._torch
        n = len(transmissions)
        interference = list(interference) if interference is not None else [0.0] * n
        budgets = [b if b is not None else self.link_budget(t, link)
                   for (t, link), b in zip(transmissions, budgets if budgets is not None else [None] * n)]
        results = [None] * n
        bits, todo = {}, []
        for k, ((t, link), budget) in enumerate(zip(transmissions, budgets)):
            if not budget.transmit:
                results[k] = self._outage(t, budget)
                continue
            bits[k] = np.asarray(self.codec.encode(t.message.payload))
            if budget.snr_db == math.inf:                 # same place: a perfect link
                results[k] = TransmissionResult(success=True, transmission=t, payload=t.message.payload,
                                                size_bits=int(bits[k].size), snr_db=math.inf, error_rate=0.0,
                                                metadata=self._metadata(budget, bit_errors=0))
            else:
                todo.append(k)
        if not todo:
            return results
        # the MCS of each transmission (fixed, or link adaptation from its SINR at each subcarrier)
        sinr = [self._sinr(budgets[k], interference[k]) for k in todo]
        _, _, info = self.adaptation(sinr, np.array([bits[k].size for k in todo]))
        groups = defaultdict(list)
        for k, i in zip(todo, info):
            groups[i['mcs'], int(bits[k].size)].append(k)
        for (mcs, num_bits), ks in groups.items():
            q, num_re, encoder, decoder, mapper, demapper = self._link(mcs, num_bits)
            padded = np.zeros((len(ks), encoder.k))      # the payload, zero-padded to the transport block
            for row, k in enumerate(ks):
                padded[row, :num_bits] = bits[k].reshape(-1)
            b = torch.as_tensor(padded, dtype=self.real, device=self.device)
            x = mapper(encoder(b))                        # (batch, resource elements)
            # the channel of each resource element: its subcarrier's, filling the subcarriers in turn
            h = np.stack([np.resize(budgets[k].response if budgets[k].response is not None else np.array([budgets[k].h]),
                                    num_re) * math.sqrt(budgets[k].power) for k in ks])
            h = torch.as_tensor(h, dtype=self.complex, device=self.device)
            var = torch.as_tensor([self.noise_power + interference[k] for k in ks], dtype=self.real, device=self.device)[:, None]
            gen = torch.Generator(device=self.device).manual_seed(int(self.rng.integers(2 ** 62)))
            noise = torch.complex(torch.randn(x.shape, generator=gen, device=self.device, dtype=self.real),
                                  torch.randn(x.shape, generator=gen, device=self.device, dtype=self.real))
            y = h * x + noise.to(self.complex) * torch.sqrt(var / 2)
            gain = (h.abs() ** 2).clamp_min(1e-30)
            llr = demapper(y / h, var / gain)             # one-tap equalizer: the noise of each resource element
            b_hat, crc = decoder(llr)
            b_hat, crc = b_hat.cpu().numpy().round().astype(np.int64), crc.cpu().numpy()
            for row, k in enumerate(ks):
                (t, link), budget = transmissions[k], budgets[k]
                decoded = b_hat[row, :num_bits].reshape(bits[k].shape)
                errors = int((decoded != bits[k]).sum())
                success = bool(crc[row])
                latency, energy = self._timing(budget, num_re, link)
                extra = {'interference_power': interference[k]} if interference[k] else {}
                results[k] = TransmissionResult(
                    success=success,
                    transmission=t,
                    payload=self.codec.decode(decoded) if success else None,
                    latency=latency,
                    size_bits=num_bits,
                    energy=energy,
                    snr_db=self._sinr_db(budget, interference[k]),
                    error_rate=errors / num_bits,
                    metadata=self._metadata(budget, mcs=mcs, resource_elements=num_re, tb_size=int(encoder.tb_size),
                                            bit_errors=errors, **extra),
                )
        return results

In [ ]:
show_doc(SionnaLinkChannel.transmit_batch)

---

[source](https://github.com/Ahmed-Khaled-Saleh/comm-core/blob/main/comm_core/sionna_phy.py#L89){target="_blank" style="float:right; font-size:smaller"}

### SionnaLinkChannel.transmit_batch

```python
def transmit_batch(
    transmissions:List[Tuple[Transmission, Link]], # Transmissions, with their links
    interference:Sequence[float] | None=None, # Power of the other transmissions at each receiver
    budgets:Sequence[LinkBudget | None] | None=None, # Their link budgets, if already computed
)->List[TransmissionResult]:
```

*Several transmissions: the ones with the same MCS and size are coded and decoded together.*

### Tests

These need Sionna, so they are flagged (`#| sionna`): `nbdev_test --flags sionna` runs them.

**The link level against the system level.** At MCS 10, 1000-bit messages are sent over a flat channel. The
fraction lost (the transport block error rate, TBLER) is compared with what `SionnaErrorModel` predicts. The PHY
abstraction should place the error-rate curve within about a dB of the simulation.

In [ ]:
from comm_core.link import UniformQuantizer
from comm_core.propagation import NoFading
from comm_core.system import ChannelInversion

In [ ]:
#| sionna
def send_all(channel, payloads, link=Link('a', 'b')):
    pairs = [(Transmission(message=Message('a', 'b', p), sender='a', receiver='b', start_time=0.0), link) for p in payloads]
    return channel.transmit_batch(pairs)

rng = np.random.default_rng(0)
codec = UniformQuantizer(bits=8, low=0, high=1)                        # 125 numbers: 1000 bits
payloads = [rng.uniform(0, 1, 125) for _ in range(200)]
model = SionnaErrorModel(mcs_index=10)
print(' SINR   simulated TBLER, 20 / 50 decoder iterations   SYS predicted')
for snr_db in (2.5, 3.0, 3.5, 4.0, 4.5, 5.0):
    lost = []
    for iterations in (20, 50):
        channel = SionnaLinkChannel(codec=codec, noise_power=10 ** (-snr_db / 10), mcs_index=10, num_bp_iter=iterations, seed=1)
        lost.append(1 - np.mean([r.success for r in send_all(channel, payloads)]))
    predicted = model([np.array([10 ** (snr_db / 10)])], np.array([1000]))[0][0]
    print(f'{snr_db:4.1f} dB {lost[0]:21.3f} / {lost[1]:.3f} {predicted:20.3f}')
    if predicted <= 0.01:
        assert lost[1] <= 0.2
    if predicted >= 0.99:
        assert lost[1] >= 0.8


 SINR   simulated TBLER, 20 / 50 decoder iterations   SYS predicted
 2.5 dB                 1.000 / 0.995                0.414
 3.0 dB                 0.980 / 0.765                0.120
 3.5 dB                 0.475 / 0.110                0.035
 4.0 dB                 0.055 / 0.010                0.009
 4.5 dB                 0.000 / 0.000                0.001
 5.0 dB                 0.000 / 0.000                0.000


* **With 50 decoder iterations,** the link level agrees with Sionna SYS where its tables have data (around 4 dB here).
* **With 20 iterations** (`TBDecoder`'s default, and so `num_bp_iter`'s), it needs about half a dB more.
* **Between table points,** the system level's curve is an interpolation. Its tables step 1.8 dB in SINR, so its
  waterfall is shallower than the real one.

**What arrives.** Codebook indices (49 of them, 8 bits each), with link adaptation: a delivered message is exactly
what was sent, and a lost one has failed its CRC.

In [ ]:
#| sionna
indices = np.arange(49) * 5
for snr_db in (0, 10, 25):
    channel = SionnaLinkChannel(noise_power=10 ** (-snr_db / 10), seed=0)          # IndexCodec(256), link adaptation
    rs = send_all(channel, [indices] * 20)
    ok = [r for r in rs if r.success]
    print(f'{snr_db:2d} dB: MCS {rs[0].metadata["mcs"]:2d}, {rs[0].metadata["resource_elements"]:4d} resource elements, '
          f'{len(ok)}/20 delivered')
    assert all((r.payload == indices).all() and r.metadata['bit_errors'] == 0 for r in ok)
    assert all(r.payload is None for r in rs if not r.success)
    assert rs[0].size_bits == 392
# reproducible with the seed; tiny payloads; an outage
a = [r.success for r in send_all(SionnaLinkChannel(noise_power=10 ** -0.35, mcs_index=10, seed=3), [indices] * 30)]
b = [r.success for r in send_all(SionnaLinkChannel(noise_power=10 ** -0.35, mcs_index=10, seed=3), [indices] * 30)]
assert a == b
r, = send_all(SionnaLinkChannel(noise_power=0.01, mcs_index=10, seed=0), [np.array([7, 200])])   # 16 bits
assert r.success and r.payload.tolist() == [7, 200]
r, = send_all(SionnaLinkChannel(fading=NoFading(0.1), power_control=ChannelInversion(10, max_power=10)), [indices])
assert not r.success and r.metadata['outage']
try:
    SionnaLinkChannel(mcs_index=0); raise AssertionError('no error')        # code rate 0.12: not encodable
except ValueError as e:
    assert '1/5' in str(e)
assert min(SionnaLinkChannel().adaptation.mcs.tolist()) == 3

 0 dB: MCS  5,  530 resource elements, 17/20 delivered
10 dB: MCS 17,  153 resource elements, 16/20 delivered
25 dB: MCS 27,   75 resource elements, 20/20 delivered


**Small messages: where Sionna SYS is wrong.** Link adaptation targets 10% losses using Sionna SYS's predictions.
At the link level, messages of a few hundred bits or more meet the target. Small messages are lost far more
often: Sionna's tables were simulated for code blocks from 24 bits, at coarse SINR steps, and are much too
optimistic for small transport blocks at high code rates. For small messages, use a fixed, lower `mcs_index`, or
check the MCS at the link level.

In [ ]:
#| sionna
rng = np.random.default_rng(0)
print('bits  SINR  MCS  REs  lost (target 0.1)')
lost = {}
for numbers in (2, 8, 49, 125):
    for snr_db in (5, 15, 25):
        channel = SionnaLinkChannel(codec=UniformQuantizer(8, 0, 1), noise_power=10 ** (-snr_db / 10), num_bp_iter=50, seed=0)
        rs = send_all(channel, [rng.uniform(0, 1, numbers) for _ in range(100)])
        lost[8 * numbers, snr_db] = 1 - np.mean([r.success for r in rs])
        print(f'{8 * numbers:4d} {snr_db:5d} {rs[0].metadata["mcs"]:4d} {rs[0].metadata["resource_elements"]:4d}  {lost[8 * numbers, snr_db]:.2f}')
assert all(lost[392, s] <= 0.2 for s in (5, 15, 25))                   # a few hundred bits: about the target

bits  SINR  MCS  REs  lost (target 0.1)
  16     5   10   13  1.00
  16    15   21    8  0.80
  16    25   26   17  1.00
  64     5   11   44  0.55
  64    15   21   18  0.35
  64    25   26   17  1.00
 392     5   11  266  0.02
 392    15   22  101  0.02
 392    25   27   75  0.00
1000     5   12  590  0.24
1000    15   22  257  0.00
1000    25   27  188  0.00


**In a network**, over a channel ray-traced at anchors with 52 subcarriers (here line-of-sight paths along a
corridor). Four nodes take turns (`TDMAProtocol`) and share the medium (`InterferenceBackend`):

In [ ]:
from comm_core import Network, Node, FullMeshTopology, TDMAProtocol, InterferenceBackend, RayTracedPathsFading, grid_points

In [ ]:
#| sionna
C, F = 299_792_458.0, 5.18e9
points = grid_points((0, 0), (30, 0), 0.5, height=1.0)
v = points[None] - points[:, None]
L = np.linalg.norm(v, axis=-1)
same = L == 0
L = np.where(same, 1.0, L)
paths = dict(a=np.where(same, 0, C / F / (4 * np.pi * L) * np.exp(-2j * np.pi * F * L / C))[..., None],
             tau=(L / C)[..., None], u_tx=(v / L[..., None])[..., None, :], u_rx=(-v / L[..., None])[..., None, :])
fading = RayTracedPathsFading(points, points, **paths, frequency=F, subcarriers=(np.arange(52) - 25.5) * 312.5e3)
nodes = ['a', 'b', 'c', 'd']
fading.set_positions({'a': (0.3, 0, 1), 'b': (6.1, 0, 1), 'c': (14.8, 0, 1), 'd': (29.6, 0, 1)})
channel = SionnaLinkChannel(fading=fading, noise_power=10 ** ((-94 - 30) / 10), seed=0)
net = Network([Node(n) for n in nodes], FullMeshTopology(nodes), TDMAProtocol(nodes, queue_size=3), InterferenceBackend(channel))
for link in (net.topology.get_link(s, r) for s in nodes for r in nodes if s != r):
    link.tx_power_w, link.bandwidth_hz = 1e-4, 20e6                     # 0.1 mW, 20 MHz
indices = np.arange(49) * 5
for step in range(4):
    for s in nodes:
        for r in nodes:
            if s != r:
                net.send(Message(s, r, indices))
    for res in net.step():
        t = res.transmission
        print(f'{t.sender} -> {t.receiver}: SINR {res.snr_db:5.1f} dB, MCS {res.metadata["mcs"]:2d}, '
              f'{res.metadata["resource_elements"]:4d} REs, delivered {res.success}')
received = net.receive('a')
assert net.metrics.summary().transmissions == 12 and all((m.payload == indices).all() for m in received)

a -> b: SINR  22.0 dB, MCS 27,   75 REs, delivered True
a -> c: SINR  14.0 dB, MCS 21,  109 REs, delivered True
a -> d: SINR   7.9 dB, MCS 14,  182 REs, delivered True
b -> a: SINR  22.0 dB, MCS 27,   75 REs, delivered True
b -> c: SINR  18.5 dB, MCS 25,   82 REs, delivered True
b -> d: SINR   9.8 dB, MCS 17,  153 REs, delivered False
c -> a: SINR  14.0 dB, MCS 21,  109 REs, delivered True
c -> b: SINR  18.5 dB, MCS 25,   82 REs, delivered True
c -> d: SINR  13.9 dB, MCS 21,  109 REs, delivered True
d -> a: SINR   7.9 dB, MCS 14,  182 REs, delivered True
d -> b: SINR   9.8 dB, MCS 17,  153 REs, delivered False
d -> c: SINR  13.9 dB, MCS 21,  109 REs, delivered True


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()